# Movie Recommendation & Analysis System
### Exploratory Data Analysis (EDA) & Insights Notebook (MySQL Edition)

This notebook connects directly to our production-compatible **MySQL database** to execute SQL queries, pull data into **Pandas DataFrames**, analyze distributions, and output visual analysis using **Matplotlib and Seaborn**.

In [ ]:
import os
import mysql.connector
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set visual style
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

# MySQL connection variables (loaded from environment variables or defaults)
db_host = os.getenv("DB_HOST", "localhost")
db_port = int(os.getenv("DB_PORT", 3306))
db_user = os.getenv("DB_USER", "root")
db_password = os.getenv("DB_PASSWORD", "password")
db_name = os.getenv("DB_NAME", "movie_platform")

try:
    conn = mysql.connector.connect(
        host=db_host,
        port=db_port,
        user=db_user,
        password=db_password,
        database=db_name
    )
    print(f"[SUCCESS] Connected to MySQL database '{db_name}' on {db_host}.")
except Exception as e:
    print(f"[ERROR] Failed to connect to MySQL database: {e}")
    print("Please verify that your MySQL server is running and database initialized (run 'python main.py --etl').")

## 1. Database Schema Verification & Raw Counts
Let's query the database tables to find the total records and inspect the schema.

In [ ]:
# Query overall size
movie_count = pd.read_sql_query("SELECT COUNT(*) as count FROM movies", conn)['count'][0]
rating_count = pd.read_sql_query("SELECT COUNT(*) as count FROM ratings", conn)['count'][0]
user_count = pd.read_sql_query("SELECT COUNT(DISTINCT user_id) as count FROM ratings", conn)['count'][0]

print(f"Database Stats:")
print(f"- Total Movies:  {movie_count:,}")
print(f"- Total Ratings: {rating_count:,}")
print(f"- Total Users:   {user_count:,}")

In [ ]:
# Inspect tables sample
print("\n--- Movies Table Sample ---")
display(pd.read_sql_query("SELECT * FROM movies LIMIT 5", conn))

print("\n--- Ratings Table Sample ---")
display(pd.read_sql_query("SELECT * FROM ratings LIMIT 5", conn))

## 2. Ratings Distribution Analysis
We query all ratings to understand user review behavior. We plot a count plot of values from 0.5 to 5.0.

In [ ]:
df_ratings = pd.read_sql_query("SELECT rating FROM ratings", conn)

plt.figure(figsize=(8, 5))
sns.countplot(data=df_ratings, x='rating', palette='viridis')
plt.title('MovieLens Rating Value Frequencies')
plt.xlabel('Rating (0.5 - 5.0)')
plt.ylabel('Number of Reviews')
plt.show()

print("Rating statistics:")
print(df_ratings['rating'].describe())

**Insight**: Ratings are left-skewed. The most frequent scores are whole stars (4.0, 3.0, 5.0) rather than half stars, indicating that users tend to think in integer increments when rating. The average rating is ~3.5.

## 3. Top 10 Highest Rated Movies (SQL Query)
We write a SQL query to retrieve movies with the highest average rating, filtering for those with at least 50 ratings to ensure statistical significance.

In [ ]:
query = """
    SELECT m.movie_id, m.title, m.genres, 
           COUNT(r.rating) as rating_count, 
           ROUND(AVG(r.rating), 2) as avg_rating
    FROM movies m
    JOIN ratings r ON m.movie_id = r.movie_id
    GROUP BY m.movie_id, m.title, m.genres
    HAVING rating_count >= 50
    ORDER BY avg_rating DESC
    LIMIT 10;
"""
df_top_movies = pd.read_sql_query(query, conn)
display(df_top_movies)

## 4. Genre Popularity & Distribution (Exploding Genre Strings)
Since genres are packed as pipe-separated values (e.g. `Action|Adventure|Sci-Fi`), we load them into pandas and split them to run aggregate counts.

In [ ]:
# Load genres and ratings
df_genres_raw = pd.read_sql_query("SELECT m.genres, r.rating FROM movies m JOIN ratings r ON m.movie_id = r.movie_id", conn)

# Explode genres
df_genres_exploded = df_genres_raw.assign(genres=df_genres_raw['genres'].str.split('|')).explode('genres')

# Compute frequency and mean rating
genre_stats = df_genres_exploded.groupby('genres').agg(
    ratings_count=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

genre_stats = genre_stats[genre_stats['genres'] != '(no genres listed)'].sort_values(by='ratings_count', ascending=False)

# Plot genre frequency
plt.figure(figsize=(12, 6))
sns.barplot(data=genre_stats.head(15), x='ratings_count', y='genres', palette='magma')
plt.title('Top 15 Genres by Frequency of Ratings')
plt.xlabel('Number of Ratings')
plt.ylabel('Genre')
plt.show()

## 5. Temporal Trends Analysis
We query timestamps and ratings to inspect how user scores have changed over the years.

In [ ]:
df_time = pd.read_sql_query("SELECT timestamp, rating FROM ratings", conn)
df_time['year'] = df_time['timestamp'].apply(lambda x: datetime.fromtimestamp(x).year)

# Group by year
yearly = df_time.groupby('year').agg(
    rating_count=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

# Filter year bounds
yearly = yearly[yearly['year'] >= 1996]

fig, ax1 = plt.subplots(figsize=(12, 5))

color = 'tab:blue'
ax1.set_xlabel('Year')
ax1.set_ylabel('Ratings Volume', color=color)
ax1.bar(yearly['year'], yearly['rating_count'], color=color, alpha=0.5)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Average Rating', color=color)
ax2.plot(yearly['year'], yearly['avg_rating'], color=color, marker='o', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Movie Ratings & Volume Trends over the Years')
plt.show()

In [ ]:
# Close database connection
if 'conn' in locals() and conn.is_connected():
    conn.close()
    print("[SUCCESS] MySQL connection closed.")